In [ ]:
!pip install torchinfo -q torchinfo

In [ ]:
!pip install datasets tokenizers torchinfo -q

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [ ]:
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

In [ ]:
DEVICE

'cuda'

In [ ]:
dataset = load_dataset("roneneldan/TinyStories", split="train[:60000]")
split_data = dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
train_texts = split_data["train"]["text"]
val_texts = split_data["test"]["text"]

In [ ]:
with open("tinystories_train.txt", "w", encoding="utf-8") as f:
    f.write("\n<eos>\n".join(train_texts))

with open("tinystories_val.txt", "w", encoding="utf-8") as f:
    f.write("\n<eos>\n".join(val_texts))

# Tokenizer

In [ ]:
tokenizer = Tokenizer(BPE(unk_token=None))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=8192,
    min_frequency=2,
    special_tokens=["<pad>", "<eos>"],
    show_progress=True
)

tokenizer.train(["tinystories_train.txt"], trainer)
tokenizer.save("tinystories_tokenizer.json")

In [ ]:
def pretokenize(input_txt_path, output_bin_path, tok):
    with open(input_txt_path, "r", encoding="utf-8") as f:
        text = f.read()
    ids = tok.encode(text).ids
    arr = np.array(ids, dtype=np.uint16)
    arr.tofile(output_bin_path)
    print(f"{input_txt_path} -> {output_bin_path} ({len(arr):,} token)")

In [ ]:
pretokenize("tinystories_train.txt", "train.bin", tokenizer)
pretokenize("tinystories_val.txt", "val.bin", tokenizer)

tinystories_train.txt -> train.bin (11,907,920 token)
tinystories_val.txt -> val.bin (1,324,761 token)


## Create DataLoaders



In [ ]:
class BPETokenizerWrapper:
    def __init__(self, tokenizer_path: str = "tinystories_tokenizer.json"):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.vocab_size = self.tokenizer.get_vocab_size()

    def encode(self, text: str) -> list:
        return self.tokenizer.encode(text).ids

    def decode(self, ids) -> str:
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        return self.tokenizer.decode(ids, skip_special_tokens=False)


In [ ]:
def load_data():
    tokenizer = BPETokenizerWrapper("tinystories_tokenizer.json")
    train_arr = np.fromfile("train.bin", dtype=np.uint16)
    val_arr = np.fromfile("val.bin", dtype=np.uint16)

    train_data = torch.from_numpy(train_arr.astype(np.int64))
    val_data = torch.from_numpy(val_arr.astype(np.int64))
    return train_data, val_data, tokenizer

In [ ]:
def get_batch(data: torch.Tensor, batch_size: int, block_size: int):
    max_start = len(data) - block_size - 1
    positions = torch.randint(high=max_start, size=(batch_size,))
    x = torch.stack([data[pos : pos + block_size] for pos in positions])
    y = torch.stack([data[pos + 1 : pos + block_size + 1] for pos in positions])
    return x, y

## Model

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim: int = 384, num_heads: int = 6, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 4),
            nn.GELU(),
            nn.Linear(embedding_dim * 4, embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor, causal_mask: torch.Tensor) -> torch.Tensor:
        x_norm = self.ln1(x)
        attn_out, _ = self.attention(
            query=x_norm, key=x_norm, value=x_norm,
            attn_mask=causal_mask, need_weights=False, is_causal=False
        )
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 384,
        num_heads: int = 6,
        num_layers: int = 6,
        dropout: float = 0.1,
        block_size: int = 256
    ):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(embedding_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output_proj = nn.Linear(embedding_dim, vocab_size, bias=False)
        self.output_proj.weight = self.token_embedding.weight  # Weight Tying

        self.loss_fn = nn.CrossEntropyLoss()
        causal_mask = torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal_mask", causal_mask)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None):
        B, T = input_ids.shape
        token_emb = self.token_embedding(input_ids)
        pos_emb = self.position_embedding(torch.arange(T, device=input_ids.device))
        x = self.dropout(token_emb + pos_emb)
        mask = self.causal_mask[:T, :T]

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_final(x)
        logits = self.output_proj(x)

        loss = None
        if targets is not None:
            loss = self.loss_fn(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int, temperature: float = 0.7):
        for _ in range(max_new_tokens):
            idx_cond = input_ids if input_ids.size(1) <= self.block_size else input_ids[:, -self.block_size:]
            logits, _ = self.forward(idx_cond)
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids

In [ ]:
EMBEDDING_DIM = 384
NUM_HEADS = 6
NUM_LAYERS = 6
BLOCK_SIZE = 256
DROPOUT = 0.1
BATCH_SIZE = 64
MAX_ITERS = 5000
EVAL_INTERVAL = 500
LEARNING_RATE = 3e-4
WARMUP_ITERS = 100

In [ ]:
def get_learning_rate(iteration):
    if iteration < WARMUP_ITERS:
        return LEARNING_RATE * (iteration / WARMUP_ITERS)
    else:
        return LEARNING_RATE
@torch.no_grad()
def evaluate(model, train_data, test_data):
    model.eval()
    results = {}
    for name, data in [("train", train_data), ("test", test_data)]:
        total_loss = 0.0
        for _ in range(30):
            x, y = get_batch(data, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE)
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, loss = model(x, y)
            total_loss += loss.item()
        results[name] = total_loss / 30
    model.train()
    return results

In [ ]:
def generate_sample(model, tokenizer, prompt: str = "Once upon a time", max_new_tokens: int = 40):
    model.eval()
    encoded = tokenizer.encode(prompt)
    input_ids = torch.tensor([encoded], dtype=torch.long, device=DEVICE)
    out_ids = model.generate(input_ids, max_new_tokens=max_new_tokens, temperature=0.7)
    decoded_text = tokenizer.decode(out_ids[0].tolist())
    model.train()
    return decoded_text

# DeepSeekMoE Transformer Block

In [ ]:
class Expert(nn.Module):
    def __init__(self, embedding_dim: int, expert_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, expert_dim),
            nn.GELU(),
            nn.Linear(expert_dim, embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class DeepSeekMoE(nn.Module):
    def __init__(
        self,
        embedding_dim: int = 384,
        expert_dim: int = 512,
        num_shared: int = 1,
        num_routed: int = 8,
        top_k: int = 2,
        dropout: float = 0.1
    ):
        super().__init__()
        self.num_routed = num_routed
        self.top_k = top_k

        self.shared_experts = nn.ModuleList([
            Expert(embedding_dim, expert_dim, dropout) for _ in range(num_shared)
        ])
        self.routed_experts = nn.ModuleList([
            Expert(embedding_dim, expert_dim, dropout) for _ in range(num_routed)
        ])
        self.router = nn.Linear(embedding_dim, num_routed, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        x_flat = x.view(-1, C)

        shared_out = torch.zeros_like(x_flat)
        for expert in self.shared_experts:
            shared_out += expert(x_flat)

        router_logits = self.router(x_flat)
        routing_weights = F.softmax(router_logits, dim=-1)
        topk_weights, topk_indices = torch.topk(routing_weights, self.top_k, dim=-1)
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)

        routed_out = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.routed_experts):
            mask = (topk_indices == i)
            if mask.any():
                token_idx = mask.any(dim=-1).nonzero(as_tuple=True)[0]
                weight = (topk_weights[token_idx] * mask[token_idx]).sum(dim=-1, keepdim=True)
                routed_out[token_idx] += weight * expert(x_flat[token_idx])

        return (shared_out + routed_out).view(B, T, C)

In [ ]:
class TransformerBlockWithMoe(nn.Module):
    def __init__(
        self,
        embedding_dim: int = 384,
        num_heads: int = 6,
        dropout: float = 0.1,
        is_moe: bool = False
    ):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.ln2 = nn.LayerNorm(embedding_dim)

        if is_moe:
            self.ffn = DeepSeekMoE(
                embedding_dim=embedding_dim,
                expert_dim=512,
                num_shared=1,
                num_routed=8,
                top_k=2,
                dropout=dropout
            )
        else:
            self.ffn = nn.Sequential(
                nn.Linear(embedding_dim, embedding_dim * 4),
                nn.GELU(),
                nn.Linear(embedding_dim * 4, embedding_dim),
                nn.Dropout(dropout)
            )

    def forward(self, x: torch.Tensor, causal_mask: torch.Tensor) -> torch.Tensor:
        x_norm = self.ln1(x)
        attn_out, _ = self.attention(
            query=x_norm, key=x_norm, value=x_norm,
            attn_mask=causal_mask, need_weights=False, is_causal=False
        )
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

In [ ]:
class GPTwithMoE(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 384,
        num_heads: int = 6,
        num_layers: int = 6,
        dropout: float = 0.1,
        block_size: int = 256
    ):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlockWithMoe(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                dropout=dropout,
                is_moe=(i > 0)
            )
            for i in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output_proj = nn.Linear(embedding_dim, vocab_size, bias=False)
        self.output_proj.weight = self.token_embedding.weight

        self.loss_fn = nn.CrossEntropyLoss()
        causal_mask = torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal_mask", causal_mask)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None):
        B, T = input_ids.shape
        token_emb = self.token_embedding(input_ids)
        pos_emb = self.position_embedding(torch.arange(T, device=input_ids.device))
        x = self.dropout(token_emb + pos_emb)
        mask = self.causal_mask[:T, :T]

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_final(x)
        logits = self.output_proj(x)

        loss = None
        if targets is not None:
            loss = self.loss_fn(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int, temperature: float = 0.7):
        for _ in range(max_new_tokens):
            idx_cond = input_ids if input_ids.size(1) <= self.block_size else input_ids[:, -self.block_size:]
            logits, _ = self.forward(idx_cond)
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids

In [ ]:
def run_experiment(model, model_name: str, train_data, test_data, tokenizer):
    print(f"\n{'='*20} {model_name} training starts {'='*20}")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total Params: {total_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    history = {"iters": [], "train_loss": [], "val_loss": []}
    start_time = time.time()

    for iteration in range(MAX_ITERS):
        lr = get_learning_rate(iteration)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        x, y = get_batch(train_data, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE)
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if iteration % EVAL_INTERVAL == 0 or iteration == MAX_ITERS - 1:
            losses = evaluate(model, train_data, test_data)
            history["iters"].append(iteration)
            history["train_loss"].append(losses["train"])
            history["val_loss"].append(losses["test"])

            elapsed = time.time() - start_time
            print(f"Iter {iteration:4d} | Train: {losses['train']:.4f} | Val: {losses['test']:.4f} | Elapsed Time: {elapsed:.1f}s")

            if iteration > 0 and iteration % 1000 == 0:
                sample = generate_sample(model, tokenizer, prompt="Once upon a time", max_new_tokens=30)
                print(f"   sample: {sample.replace(chr(10), ' ')}")

    return history

In [24]:
train_data, val_data, tokenizer = load_data()
vocab_size = tokenizer.vocab_size

# dense baseline GPT
dense_model = GPT(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    block_size=BLOCK_SIZE
).to(DEVICE)

dense_history = run_experiment(dense_model, "Dense Baseline GPT", train_data, val_data, tokenizer)

# DeepSeekMoE GPT
moe_model = GPTwithMoE(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    block_size=BLOCK_SIZE
).to(DEVICE)

moe_history = run_experiment(moe_model, "DeepSeekMoE GPT", train_data, val_data, tokenizer)


==================== Dense Baseline GPT training starts ====================
Total Params: 13,891,584
Iter    0 | Train: 9.0287 | Val: 9.0281 | Elapsed Time: 5.8s
Iter  500 | Train: 3.0658 | Val: 3.0824 | Elapsed Time: 148.4s
Iter 1000 | Train: 2.5457 | Val: 2.5831 | Elapsed Time: 293.1s
   sample: Once upon a time, there was a little girl named Lily. She loved to play outside in the sky. One day, she found a shiny jar of yummy clothes.
Iter 1500 | Train: 2.3009 | Val: 2.3395 | Elapsed Time: 438.2s
Iter 2000 | Train: 2.1517 | Val: 2.2024 | Elapsed Time: 583.2s
   sample: Once upon a time, there was a little girl named Lily. She loved to play outside in the park. One day, she saw a shiny stone on the ground.
Iter 2500 | Train: 2.0292 | Val: 2.1006 | Elapsed Time: 728.4s
Iter 3000 | Train: 1.9738 | Val: 2.0519 | Elapsed Time: 873.3s
   sample: Once upon a time, there was a little girl named Lily. She loved to play outside, but she didn't know what to do. One day, she saw her
Iter 3500 |